# Handling Missing Values Before Exporting

<a href="https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week07/06.Handling-Missing-Values-Before-Exporting/notebooks/06-handling-missing-values-before-exporting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview
In real-world data collection, sensor errors, transmission drops, and missing logs result in **missing values** (`NaN` or `None`).
Exporting datasets with unhandled missing values can cause crashes in downstream databases, break machine learning pipelines, or skew statistical reports.

In this notebook, you will learn how to detect, inspect, drop, and impute (fill) missing values in Pandas, followed by validation checks before exporting clean data.

## 1. Load Weather Dataset
Let's load `sample_weather.csv` (which contains missing weather sensor readings).

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('sample_weather.csv', parse_dates=['Date'])
display(df.head(8))

## 2. Detecting Missing Values: isna() and sum()
Use `.isna()` (or `.isnull()`) combined with `.sum()` to identify missing entries per column.

In [ ]:
print('Count of missing values per column:')
display(df.isna().sum())

print('\nPercentage of missing values:')
display((df.isna().mean() * 100).round(2).astype(str) + '%')

print('\nRows with missing values:')
display(df[df.isna().any(axis=1)])

## 3. Strategy A: Dropping Missing Values (.dropna())
When missing records represent corrupted or unrecoverable entries, use `.dropna()`.

In [ ]:
# Drop rows containing any null value
df_dropped = df.dropna()
print(f"Original row count: {len(df)}")
print(f"Row count after dropna(): {len(df_dropped)}")
display(df_dropped)

## 4. Strategy B: Imputing / Filling Missing Values (.fillna())
Dropping rows discards other valuable columns. Instead, we can impute missing values:
- **Constant fill**: e.g., missing rainfall replaced with `0.0`.
- **Statistical fill**: missing temperature replaced with column mean or median.

In [ ]:
df_imputed = df.copy()
mean_temp = df_imputed['Temperature'].mean()
print(f"Computed mean temperature: {mean_temp:.2f}°C")

df_imputed['Temperature'] = df_imputed['Temperature'].fillna(mean_temp)
print('Missing counts after filling:')
display(df_imputed.isna().sum())

## 5. Validating and Exporting Clean Data
**Best Practice**: Validate that zero missing values remain before exporting to disk.

In [ ]:
# Assertion check
total_nulls = df_imputed.isna().sum().sum()
assert total_nulls == 0, f"Error: {total_nulls} nulls remaining!"
print('Validation check: PASSED (0 missing values remaining)')

# Export cleanly
output_file = 'cleaned_weather_export.csv'
df_imputed.to_csv(output_file, index=False, float_format='%.2f')
print(f"Saved clean dataset to '{output_file}'")
display(pd.read_csv(output_file).head(8))

## Enrichment
### Group-Specific Imputation
Impute missing values using the mean of each specific city:
```python
df['Temperature'] = df.groupby('City')['Temperature'].transform(
    lambda grp: grp.fillna(grp.mean())
)
```

### Forward-fill in Time-Series
Propagate last observed temperature forward:
```python
df['Temperature'] = df['Temperature'].ffill()
```

## Takeaways
- Always audit missing values using `df.isna().sum()` immediately after ingestion.
- Choose between dropping (`.dropna()`) and imputing (`.fillna()`) based on domain requirements.
- When imputing, choose sensible defaults (e.g. `0.0` for rainfall, column mean/median for temperature).
- Validate data hygiene with assertions (`assert df.isna().sum().sum() == 0`) before exporting.
- Always export with `index=False`.

## Conclusion
Handling missing values proactively prevents subtle calculation biases and ensures data products are reliable and production-ready.

## Exercises
**Exercise 1:** Count the number of non-null values in the dataset using `.notna().sum()`.

**Exercise 2:** Create a copy of `df` and fill missing temperature using the **median** temperature.

**Exercise 3:** Use forward fill (`.ffill()`) on a copy of `df` and confirm that row index 6 inherits the temperature of row 5.

In [ ]:
# Write your practice code here

# --- Solutions ---
# print('Exercise 1: Non-null counts:')
# display(df.notna().sum())
#
# df_median = df.copy()
# df_median['Temperature'] = df_median['Temperature'].fillna(df_median['Temperature'].median())
# print('Exercise 2: Null count =', df_median['Temperature'].isna().sum())
#
# df_ffill = df.copy()
# df_ffill['Temperature'] = df_ffill['Temperature'].ffill()
# print('Exercise 3: Row 6 temperature =', df_ffill.loc[6, 'Temperature'])